In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
from scipy import sparse
from pathlib import Path

In [2]:
adata_nk = sc.read_h5ad("/scratch/user/s4575250/BIOX7014_Thesis/write/03_batch_expression/PICA_Batch001-Batch007/PICA_Batch001-Batch007_nk_combined_annot_with_age_scvi.h5ad")

In [3]:
adata_nk

AnnData object with n_obs × n_vars = 187173 × 38606
    obs: 'status', 'assignment', 'pica_id', 'pool_id', 'sequencing_batch', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'status_manual', 'cell_type', 'broad_cell_type', 'pica_broad_cell_type', 'pica_cell_type', 'pica_cell_type_01_myeloid', 'pica_cell_type_02_b', 'pica_cell_type_03_nk_t', 'pica_cell_type_complete', 'pica_broad_cell_type_complete', 'pica_broad_cell_type_complete_01_gdT_corrected', 'pica_cell_type_complete_01_gdT_corrected', 'status_manual_01_pica', 'Record ID', 'Sex', 'Age_years', 'Age', 'Event Name', 'Was baseline blood sample collected?', 'IFC ID', "Child's sex", 'Age in 

In [4]:
# set metadata column
donor_col = "pica_id"
celltype_col = "pica_cell_type_complete_01_gdT_corrected"
age_col = "Age_years"
sex_col = "Sex"
batch_col = "sequencing_batch"

In [5]:
print(adata_nk.shape)
print(adata_nk.obs[celltype_col].value_counts())

(187173, 38606)
pica_cell_type_complete_01_gdT_corrected
CD56_dim CD16+ NK           171221
CD56_dim CD16+ KLRC2+ NK     15081
CD56_bright CD16- NK           871
Name: count, dtype: int64


In [6]:
# Subset metadata and layers only
obs = adata_nk.obs.copy()

nk_mask = obs[celltype_col].astype(str).isin([
    "CD56_dim CD16+ NK",
    "CD56_dim CD16+ KLRC2+ NK",
    "CD56_bright CD16- NK"
])

required_cols = [donor_col, celltype_col, age_col, sex_col, batch_col]
valid_mask = obs[required_cols].notna().all(axis=1)

keep_mask = nk_mask & valid_mask

obs_nk = obs.loc[keep_mask].copy()

print(obs_nk.shape)
print(obs_nk[celltype_col].value_counts())

(187173, 78)
pica_cell_type_complete_01_gdT_corrected
CD56_dim CD16+ NK           171221
CD56_dim CD16+ KLRC2+ NK     15081
CD56_bright CD16- NK           871
Name: count, dtype: int64


In [7]:
# extract raw counts
X = adata_nk.layers["counts"]

if sparse.issparse(X):
    X = X.tocsr()

genes = adata_nk.var_names.astype(str)

# Get original integer positions of kept CD4 cells
cell_positions = np.where(keep_mask.values)[0]

print(X.shape)

(187173, 38606)


## Create Pseudobulk

In [8]:
# donor × cell type
obs_nk["pseudobulk_id"] = (
    obs_nk[donor_col].astype(str) + "__" + obs_nk[celltype_col].astype(str)
)

# number of pseudobulk samples
print(obs_nk["pseudobulk_id"].nunique())
print(obs_nk["pseudobulk_id"].value_counts().describe())

295
count     295.000000
mean      634.484746
std       817.025440
min         4.000000
25%        26.000000
50%       154.000000
75%      1128.000000
max      5519.000000
Name: count, dtype: float64


In [9]:
# Aggregate raw counts
pb_counts = []
pb_meta = []

for pb_id, group_index in obs_nk.groupby("pseudobulk_id").indices.items():
    
    # group_index = positions within obs_nk
    # original_positions = positions within original adata/X
    original_positions = cell_positions[group_index]
    meta_sub = obs_nk.iloc[group_index]
    
    summed_counts = np.asarray(X[original_positions, :].sum(axis=0)).ravel()
    
    pb_counts.append(summed_counts)
    
    pb_meta.append({
        "sample_id": pb_id,
        "donor_id": meta_sub[donor_col].iloc[0],
        "cell_type": meta_sub[celltype_col].iloc[0],
        "age": meta_sub[age_col].iloc[0],
        "sex": meta_sub[sex_col].iloc[0],
        "batch": meta_sub[batch_col].iloc[0],
        "n_cells": len(original_positions)
    })

pb_counts_df = pd.DataFrame(
    pb_counts,
    index=[m["sample_id"] for m in pb_meta],
    columns=genes
)

pb_meta_df = pd.DataFrame(pb_meta).set_index("sample_id")

print("Pseudobulk count matrix:")
print(pb_counts_df.shape)

print("\nPseudobulk metadata:")
print(pb_meta_df.shape)

print("\nPseudobulk samples per cell type:")
print(pb_meta_df["cell_type"].value_counts())

Pseudobulk count matrix:
(295, 38606)

Pseudobulk metadata:
(295, 6)

Pseudobulk samples per cell type:
cell_type
CD56_dim CD16+ KLRC2+ NK    128
CD56_dim CD16+ NK           128
CD56_bright CD16- NK         39
Name: count, dtype: int64


In [12]:
# filter low-cell pseudobulk samples
min_cells = 20

keep_samples = pb_meta_df["n_cells"] >= min_cells

pb_counts_df = pb_counts_df.loc[keep_samples]
pb_meta_df = pb_meta_df.loc[keep_samples]

print("After filtering pseudobulk samples with n_cells <", min_cells)
print("Counts:", pb_counts_df.shape)
print("Metadata:", pb_meta_df.shape)

print("\nSamples per cell type after filtering:")
print(pb_meta_df["cell_type"].value_counts())

print("\nCell counts per cell type after filtering:")
print(pb_meta_df.groupby("cell_type")["n_cells"].sum())

After filtering pseudobulk samples with n_cells < 20
Counts: (250, 38606)
Metadata: (250, 6)

Samples per cell type after filtering:
cell_type
CD56_dim CD16+ NK           128
CD56_dim CD16+ KLRC2+ NK    106
CD56_bright CD16- NK         16
Name: count, dtype: int64

Cell counts per cell type after filtering:
cell_type
CD56_bright CD16- NK           570
CD56_dim CD16+ KLRC2+ NK     14775
CD56_dim CD16+ NK           171221
Name: n_cells, dtype: int64


In [13]:
pb_counts_df.T.to_csv("/scratch/user/s4575250/BIOX7014_Thesis/write/04_DEG/PICA_Batch001-Batch007/nk_pseudobulk_counts_gene_by_sample.csv")
pb_meta_df.to_csv("/scratch/user/s4575250/BIOX7014_Thesis/write/04_DEG/PICA_Batch001-Batch007/nk_pseudobulk_metadata.csv")

In [14]:
# Sanity checks
counts_check = pd.read_csv("/scratch/user/s4575250/BIOX7014_Thesis/write/04_DEG/PICA_Batch001-Batch007/nk_pseudobulk_counts_gene_by_sample.csv", index_col=0)
meta_check = pd.read_csv("/scratch/user/s4575250/BIOX7014_Thesis/write/04_DEG/PICA_Batch001-Batch007/nk_pseudobulk_metadata.csv", index_col=0)

print("Counts check:", counts_check.shape)
print("Metadata check:", meta_check.shape)

assert list(counts_check.columns) == list(meta_check.index)

print("Sample names match between counts and metadata.")
print(meta_check.head())

Counts check: (38606, 250)
Metadata check: (250, 6)
Sample names match between counts and metadata.
                                    donor_id                 cell_type    age  \
sample_id                                                                       
PICA0001__CD56_dim CD16+ KLRC2+ NK  PICA0001  CD56_dim CD16+ KLRC2+ NK   2.09   
PICA0001__CD56_dim CD16+ NK         PICA0001         CD56_dim CD16+ NK   2.09   
PICA0002__CD56_dim CD16+ KLRC2+ NK  PICA0002  CD56_dim CD16+ KLRC2+ NK  15.09   
PICA0002__CD56_dim CD16+ NK         PICA0002         CD56_dim CD16+ NK  15.09   
PICA0003__CD56_dim CD16+ KLRC2+ NK  PICA0003  CD56_dim CD16+ KLRC2+ NK   0.83   

                                     sex  \
sample_id                                  
PICA0001__CD56_dim CD16+ KLRC2+ NK  Male   
PICA0001__CD56_dim CD16+ NK         Male   
PICA0002__CD56_dim CD16+ KLRC2+ NK  Male   
PICA0002__CD56_dim CD16+ NK         Male   
PICA0003__CD56_dim CD16+ KLRC2+ NK  Male   

                       